In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.isnull().any()
#no missing data

Pregnancies                 False
Glucose                     False
BloodPressure               False
SkinThickness               False
Insulin                     False
BMI                         False
DiabetesPedigreeFunction    False
Age                         False
Outcome                     False
dtype: bool

In [6]:
df.isna().sum()
#no NaN data
#if you like, you can also use df.isnull().any()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [8]:
df. describe()
#the fact that the value of 'count' under each column is 768 tells us that there is no missing(NOT NaN) value

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [9]:
# we assume that there is no outlier

In [13]:
df.Outcome.value_counts()

0    500
1    268
Name: Outcome, dtype: int64

In [15]:
X = df.drop('Outcome', axis=1)
y = df.Outcome

In [16]:
#looking at the min vs max values for each column, we should do scaling
#though the values are not too..., yet let's do it to be on the safer side

In [17]:
from sklearn.preprocessing import StandardScaler

In [18]:
scale = StandardScaler()

In [22]:
X_scaled = scale.fit_transform(X)
X_scaled[:3]
#first three rows

array([[ 0.63994726,  0.84832379,  0.14964075,  0.90726993, -0.69289057,
         0.20401277,  0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575,  0.53090156, -0.69289057,
        -0.68442195, -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, -1.28821221, -0.69289057,
        -1.10325546,  0.60439732, -0.10558415]])

In [27]:
df.Outcome.value_counts()

0    500
1    268
Name: Outcome, dtype: int64

In [28]:
268/500
#the ratio of 0 to 1 i.e. ratio of those with and without diabetes is somehow(though not too much and can be neglected)not balanced
#like double the number of those who have is the number of those who don't have
#that's why stratify=y below

0.536

In [29]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, stratify=y, random_state=10)
#we set stratify as y so that the ratio of x to y in our training data will be balanced

In [31]:
X_train.shape

(576, 8)

In [32]:
X_test.shape

(192, 8)

In [33]:
from sklearn.tree import DecisionTreeClassifier
#you can use any model classifier you want

In [34]:
DTC = DecisionTreeClassifier()

In [35]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

scores = cross_val_score(DecisionTreeClassifier(), X, y, cv=5)
scores

array([0.7012987 , 0.65584416, 0.68181818, 0.80392157, 0.7254902 ])

In [36]:
scores.mean()

0.7136745607333843

In [38]:
from sklearn.ensemble import BaggingClassifier

In [52]:
BC = BaggingClassifier(
    base_estimator=DecisionTreeClassifier(),
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=0
)

In [53]:
BC.fit(X_train, y_train)

BaggingClassifier(base_estimator=DecisionTreeClassifier(), max_samples=0.8,
                  n_estimators=100, oob_score=True, random_state=0)

In [54]:
BC.oob_score_

0.7534722222222222

In [55]:
BC.score(X_test, y_test)

0.7760416666666666

In [56]:
scores = cross_val_score(BC, X, y, cv=5)
scores.mean()
#using just decision tree ALONE initially, the mean of our cross val score was 71...., but now with the baggging model, the
#decision tree now gives 75....

0.7578728461081402

In [57]:
#for unsatble sclassifiers like decision tree, bagging helps to improve accuracy

In [58]:
# lets try random forest too:
from sklearn.ensemble import RandomForestClassifier

In [60]:
score = cross_val_score(RandomForestClassifier(), X,y, cv=5)
score.mean()

0.7722179781003311

In [62]:
# now lets use the random forest with bagging model:

BC_RFC = BaggingClassifier(
    base_estimator=RandomForestClassifier(),
    n_estimators=100,
    max_samples=0.8,
    oob_score=True,
    random_state=0
)

In [64]:
score = cross_val_score(BC_RFC, X,y, cv=5)
score.mean()

0.7643918173329938

In [ ]:
#bagging does improve the accuracy of Random forest(here)